In [1]:
import pandas as pd
import sqlite3
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
conn = sqlite3.connect("../data/bank_loan.db")
df = pd.read_sql("SELECT * FROM clean_data", conn)
conn.close()

print("Shape:", df.shape)
df.head()

Shape: (5000, 12)


,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,25,1,49,4,0.0,1,0,0,1,0,0,0
1,45,19,34,3,0.0,1,0,0,1,0,0,0
2,39,15,11,1,0.0,1,0,0,0,0,0,0
3,35,9,100,1,0.0,2,0,0,0,0,0,0
4,35,8,45,4,0.0,2,0,0,0,0,0,1


In [3]:
# X = df.drop(columns=["Personal Loan"])
X = df.drop(columns=["Personal Loan", "CCAvg", "Securities Account", "Online", "CreditCard"])
y = df["Personal Loan"]

print("Features (X):", X.columns.tolist())
print("Target  (y) : Personal Loan")
print("X shape:", X.shape)
print("y shape:", y.shape)

Features (X): ['Age', 'Experience', 'Income', 'Family', 'Education', 'Mortgage', 'CD Account']
Target  (y) : Personal Loan
X shape: (5000, 7)
y shape: (5000,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train)} rows")
print(f"Test size  : {len(X_test)} rows")

Train size : 4000 rows
Test size  : 1000 rows


In [5]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Scaling done ✅")

Scaling done ✅


In [6]:
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_sc, y_train)

print("Model trained ✅")

Model trained ✅


In [7]:
y_pred = model.predict(X_test_sc)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 98.60%


In [8]:
print(classification_report(y_test, y_pred, target_names=["No Loan", "Loan"]))

              precision    recall  f1-score   support

     No Loan       0.99      0.99      0.99       904
        Loan       0.95      0.91      0.93        96

    accuracy                           0.99      1000
   macro avg       0.97      0.95      0.96      1000
weighted avg       0.99      0.99      0.99      1000



In [9]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(f"  Correctly predicted No Loan : {cm[0][0]}")
print(f"  Wrongly predicted Loan      : {cm[0][1]}")
print(f"  Missed actual Loan          : {cm[1][0]}")
print(f"  Correctly predicted Loan    : {cm[1][1]}")

Confusion Matrix:
  Correctly predicted No Loan : 899
  Wrongly predicted Loan      : 5
  Missed actual Loan          : 9
  Correctly predicted Loan    : 87


In [10]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("Feature Importance:")
for feat, score in importance.items():
    bar = "█" * int(score * 100)
    print(f"  {feat:<22} {score*100:5.1f}%  {bar}")

Feature Importance:
  Income                  59.2%  ███████████████████████████████████████████████████████████
  Education               11.1%  ███████████
  Family                   7.9%  ███████
  CD Account               7.3%  ███████
  Mortgage                 5.1%  █████
  Age                      4.7%  ████
  Experience               4.7%  ████


In [11]:
joblib.dump(model,               "../models/loan_model.pkl")
joblib.dump(scaler,              "../models/scaler.pkl")
joblib.dump(X.columns.tolist(),  "../models/feature_names.pkl")

print("✅ Model saved  → models/loan_model.pkl")
print("✅ Scaler saved → models/scaler.pkl")

✅ Model saved  → models/loan_model.pkl
✅ Scaler saved → models/scaler.pkl
